# 🥇 Gold Layer - Inventory & Supply Chain Analytics
## AWS S3 External Storage via Unity Catalog

**Purpose**: Create business-ready inventory, supply chain, and profitability analytics

**Source**: `workspace.`workspace-adventureworks-silver`` (S3)
**Target**: `workspace.`workspace-adventureworks-gold`` (S3)

**Gold Tables to Create**:
1. **gold_inventory_turnover** - Inventory turnover metrics and stock health by product
2. **gold_vendor_performance** - Vendor performance: delivery, quality, and lead times
3. **gold_product_profitability** - Product profitability with accurate COGS from purchases
4. **gold_reorder_alerts** - Stock level monitoring and reorder recommendations

**Business Metrics**:
- Inventory turnover rate and days on hand
- Vendor on-time delivery and quality scores
- Product gross margin with purchase-based COGS
- Stock level alerts and reorder points
- Lead time analysis for supply chain optimization

**Addresses Problem Statement**:
- Track inventory turnover and prevent stockouts
- Vendor performance and lead time analysis
- Calculate accurate product profitability including purchase costs
- Supply chain optimization with reorder point alerts

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import datetime
import time
import pyspark.sql.functions as F

# Configuration
SILVER_SCHEMA = "`workspace-adventureworks-silver`"
GOLD_SCHEMA = "`workspace-adventureworks-gold`"

print("=" * 80)
print("🥇 GOLD LAYER - INVENTORY & SUPPLY CHAIN ANALYTICS")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print(f"Source: workspace.{SILVER_SCHEMA} (AWS S3)")
print(f"Target: workspace.{GOLD_SCHEMA} (AWS S3)")
print("=" * 80)
print()

transformation_results = []

In [0]:
# Inventory Turnover Analytics
print("\n" + "=" * 80)
print("📦 BUILDING gold_inventory_turnover")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_sales = spark.table(f"workspace.{SILVER_SCHEMA}.fact_sales")
    fact_inventory = spark.table(f"workspace.{SILVER_SCHEMA}.fact_inventory")
    dim_product = spark.table(f"workspace.{SILVER_SCHEMA}.dim_product").filter("is_current = true")
    
    # Calculate sales volume by product and year
    sales_volume = (fact_sales
        .withColumn("sales_year", F.year(col("order_date")))
        .groupBy("product_id", "sales_year")
        .agg(
            F.sum("order_quantity").alias("total_units_sold"),
            F.sum("line_total").alias("total_sales_revenue"),
            F.avg("unit_price").alias("avg_selling_price")
        )
    )
    
    # Calculate inventory activity by product and year
    inventory_activity = (fact_inventory
        .withColumn("inventory_year", F.year(col("transaction_date")))
        .groupBy("product_id", "inventory_year")
        .agg(
            # Count different transaction types
            F.sum(F.when(col("transaction_type") == "W", col("quantity")).otherwise(0)).alias("work_order_in"),
            F.sum(F.when(col("transaction_type") == "S", col("quantity")).otherwise(0)).alias("sales_out"),
            F.sum(F.when(col("transaction_type") == "P", col("quantity")).otherwise(0)).alias("purchase_in"),
            F.sum(F.when(col("transaction_type").isin(["W", "P"]), col("quantity")).otherwise(0)).alias("total_in"),
            F.sum(F.when(col("transaction_type") == "S", col("quantity")).otherwise(0)).alias("total_out"),
            F.avg(col("actual_cost")).alias("avg_inventory_cost"),
            F.count("transaction_id").alias("transaction_count")
        )
    )
    
    # Join with product dimension and combine metrics
    gold_inventory_turnover = (sales_volume
        .join(
            inventory_activity,
            (sales_volume.product_id == inventory_activity.product_id) & 
            (sales_volume.sales_year == inventory_activity.inventory_year),
            "left"
        )
        .join(dim_product, sales_volume.product_id == dim_product.product_id)
        .select(
            sales_volume.product_id,
            col("product_name"),
            col("product_number"),
            col("category_name"),
            col("subcategory_name"),
            col("standard_cost"),
            col("list_price"),
            col("sales_year"),
            col("total_units_sold"),
            col("total_sales_revenue"),
            col("avg_selling_price"),
            F.coalesce(col("work_order_in"), F.lit(0)).alias("work_order_in"),
            F.coalesce(col("sales_out"), F.lit(0)).alias("sales_out"),
            F.coalesce(col("purchase_in"), F.lit(0)).alias("purchase_in"),
            F.coalesce(col("total_in"), F.lit(0)).alias("total_stock_in"),
            F.coalesce(col("total_out"), F.lit(0)).alias("total_stock_out"),
            F.coalesce(col("avg_inventory_cost"), col("standard_cost")).alias("avg_inventory_cost"),
            F.coalesce(col("transaction_count"), F.lit(0)).alias("inventory_transactions")
        )
        # Calculate derived metrics
        .withColumn("net_stock_change", col("total_stock_in") - col("total_stock_out"))
        .withColumn("avg_stock_on_hand", 
            (col("total_stock_in") + col("total_stock_out")) / 2
        )
        .withColumn("inventory_turnover_ratio",
            F.when(col("avg_stock_on_hand") > 0,
                col("total_units_sold") / col("avg_stock_on_hand")
            ).otherwise(None)
        )
        .withColumn("days_inventory_on_hand",
            F.when(col("inventory_turnover_ratio").isNotNull() & (col("inventory_turnover_ratio") > 0),
                365 / col("inventory_turnover_ratio")
            ).otherwise(None)
        )
        .withColumn("stock_health_status",
            F.when(col("inventory_turnover_ratio").isNull(), "No Data")
            .when(col("inventory_turnover_ratio") > 12, "Fast Moving")
            .when(col("inventory_turnover_ratio") > 6, "Healthy")
            .when(col("inventory_turnover_ratio") > 2, "Slow Moving")
            .otherwise("Stagnant")
        )
    )
    
    # Write to Gold
    (gold_inventory_turnover
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.{GOLD_SCHEMA}.gold_inventory_turnover")
    )
    
    duration = time.time() - start_time
    row_count = gold_inventory_turnover.count()
    
    transformation_results.append({
        "table": "gold_inventory_turnover",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_inventory_turnover created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    duration_on_error = time.time() - start_time
    error_msg = str(e)
    print(f"\n❌ Failed: {error_msg}")
    transformation_results.append({
        "table": "gold_inventory_turnover",
        "status": "failed",
        "row_count": 0,
        "duration_seconds": round(duration_on_error, 2)
    })

In [0]:
# Vendor Performance Analytics
print("\n" + "=" * 80)
print("🚚 BUILDING gold_vendor_performance")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_purchases = spark.table(f"workspace.{SILVER_SCHEMA}.fact_purchases")
    dim_vendor = spark.table(f"workspace.{SILVER_SCHEMA}.dim_vendor").filter("is_current = true")
    
    # Calculate vendor performance metrics
    vendor_metrics = (fact_purchases
        .withColumn("purchase_year", F.year(col("order_date")))
        .withColumn("lead_time_days", 
            F.datediff(col("ship_date"), col("order_date"))
        )
        .withColumn("delivery_delay_days",
            F.datediff(col("ship_date"), col("due_date"))
        )
        .withColumn("is_on_time",
            F.when(col("ship_date") <= col("due_date"), 1).otherwise(0)
        )
        .withColumn("is_late",
            F.when(col("ship_date") > col("due_date"), 1).otherwise(0)
        )
        .withColumn("quality_acceptance_rate",
            F.when(col("received_quantity") > 0,
                (col("received_quantity") - col("rejected_quantity")) / col("received_quantity") * 100
            ).otherwise(None)
        )
        .groupBy("vendor_id", "purchase_year")
        .agg(
            # Order metrics
            F.countDistinct("purchase_order_id").alias("total_purchase_orders"),
            F.count("purchase_order_detail_id").alias("total_line_items"),
            
            # Financial metrics
            F.sum("line_total").alias("total_purchase_amount"),
            F.sum("order_subtotal").alias("total_order_subtotal"),
            F.sum("freight").alias("total_freight_cost"),
            F.avg("unit_price").alias("avg_unit_price"),
            
            # Quantity metrics
            F.sum("order_quantity").alias("total_ordered_quantity"),
            F.sum("received_quantity").alias("total_received_quantity"),
            F.sum("rejected_quantity").alias("total_rejected_quantity"),
            F.sum("stocked_quantity").alias("total_stocked_quantity"),
            
            # Delivery performance
            F.avg("lead_time_days").alias("avg_lead_time_days"),
            F.min("lead_time_days").alias("min_lead_time_days"),
            F.max("lead_time_days").alias("max_lead_time_days"),
            F.stddev("lead_time_days").alias("lead_time_std_dev"),
            
            # On-time delivery
            F.sum("is_on_time").alias("on_time_deliveries"),
            F.sum("is_late").alias("late_deliveries"),
            F.avg(F.when(col("is_late") == 1, col("delivery_delay_days")).otherwise(None)).alias("avg_delay_when_late"),
            
            # Quality metrics
            F.avg("quality_acceptance_rate").alias("avg_quality_acceptance_rate"),
            
            # Product diversity
            F.countDistinct("product_id").alias("unique_products_supplied")
        )
        # Calculate derived percentages
        .withColumn("on_time_delivery_rate",
            (col("on_time_deliveries") / (col("on_time_deliveries") + col("late_deliveries")) * 100)
        )
        .withColumn("rejection_rate",
            F.when(col("total_received_quantity") > 0,
                col("total_rejected_quantity") / col("total_received_quantity") * 100
            ).otherwise(0)
        )
        .withColumn("fulfillment_rate",
            F.when(col("total_ordered_quantity") > 0,
                col("total_received_quantity") / col("total_ordered_quantity") * 100
            ).otherwise(0)
        )
    )
    
    # Join with vendor dimension
    gold_vendor_performance = (vendor_metrics
        .join(dim_vendor, "vendor_id")
        .select(
            col("vendor_id"),
            col("vendor_name"),
            col("account_number"),
            col("credit_rating"),
            col("is_preferred_vendor"),
            col("is_active"),
            col("purchase_year"),
            col("total_purchase_orders"),
            col("total_line_items"),
            col("total_purchase_amount"),
            col("total_order_subtotal"),
            col("total_freight_cost"),
            col("avg_unit_price"),
            col("total_ordered_quantity"),
            col("total_received_quantity"),
            col("total_rejected_quantity"),
            col("total_stocked_quantity"),
            col("avg_lead_time_days"),
            col("min_lead_time_days"),
            col("max_lead_time_days"),
            col("lead_time_std_dev"),
            col("on_time_deliveries"),
            col("late_deliveries"),
            col("on_time_delivery_rate"),
            col("avg_delay_when_late"),
            col("avg_quality_acceptance_rate"),
            col("rejection_rate"),
            col("fulfillment_rate"),
            col("unique_products_supplied")
        )
        # Add overall vendor performance score
        .withColumn("vendor_performance_score",
            (col("on_time_delivery_rate") * 0.4 + 
             col("avg_quality_acceptance_rate") * 0.3 + 
             col("fulfillment_rate") * 0.3)
        )
        .withColumn("vendor_grade",
            F.when(col("vendor_performance_score") >= 95, "A - Excellent")
            .when(col("vendor_performance_score") >= 85, "B - Good")
            .when(col("vendor_performance_score") >= 75, "C - Average")
            .when(col("vendor_performance_score") >= 65, "D - Below Average")
            .otherwise("F - Poor")
        )
    )
    
    # Write to Gold
    (gold_vendor_performance
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.{GOLD_SCHEMA}.gold_vendor_performance")
    )
    
    duration = time.time() - start_time
    row_count = gold_vendor_performance.count()
    
    transformation_results.append({
        "table": "gold_vendor_performance",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_vendor_performance created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({
        "table": "gold_vendor_performance",
        "status": "failed",
        "row_count": 0,
        "duration_seconds": round(time.time() - start_time, 2)
    })

In [0]:
# Product Profitability with Accurate COGS
print("\n" + "=" * 80)
print("💰 BUILDING gold_product_profitability")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_sales = spark.table(f"workspace.{SILVER_SCHEMA}.fact_sales")
    fact_purchases = spark.table(f"workspace.{SILVER_SCHEMA}.fact_purchases")
    dim_product = spark.table(f"workspace.{SILVER_SCHEMA}.dim_product").filter("is_current = true")
    
    # Calculate sales metrics by product and year
    sales_metrics = (fact_sales
        .withColumn("sales_year", F.year(col("order_date")))
        .groupBy("product_id", "sales_year")
        .agg(
            F.count("sales_order_id").alias("sales_order_count"),
            F.sum("order_quantity").alias("total_units_sold"),
            F.sum("line_total").alias("total_sales_revenue"),
            F.avg("unit_price").alias("avg_selling_price"),
            F.min("unit_price").alias("min_selling_price"),
            F.max("unit_price").alias("max_selling_price"),
            F.avg("unit_price_discount").alias("avg_discount_pct"),
            F.countDistinct("customer_id").alias("unique_customers")
        )
    )
    
    # Calculate purchase-based COGS by product and year
    purchase_cogs = (fact_purchases
        .withColumn("purchase_year", F.year(col("order_date")))
        .groupBy("product_id", "purchase_year")
        .agg(
            F.count("purchase_order_id").alias("purchase_order_count"),
            F.sum("order_quantity").alias("total_units_purchased"),
            F.sum("line_total").alias("total_purchase_cost"),
            F.avg("unit_price").alias("avg_purchase_unit_cost"),
            F.min("unit_price").alias("min_purchase_unit_cost"),
            F.max("unit_price").alias("max_purchase_unit_cost"),
            F.stddev("unit_price").alias("purchase_cost_std_dev"),
            F.countDistinct("vendor_id").alias("vendor_count")
        )
    )
    
    # Join sales with purchases and product dimension
    gold_product_profitability = (sales_metrics
        .join(
            purchase_cogs,
            (sales_metrics.product_id == purchase_cogs.product_id) & 
            (sales_metrics.sales_year == purchase_cogs.purchase_year),
            "left"
        )
        .join(dim_product, sales_metrics.product_id == dim_product.product_id)
        .select(
            sales_metrics.product_id,
            col("product_name"),
            col("product_number"),
            col("category_name"),
            col("subcategory_name"),
            col("standard_cost"),
            col("list_price"),
            col("sales_year"),
            
            # Sales metrics
            col("sales_order_count"),
            col("total_units_sold"),
            col("total_sales_revenue"),
            col("avg_selling_price"),
            col("min_selling_price"),
            col("max_selling_price"),
            col("avg_discount_pct"),
            col("unique_customers"),
            
            # Purchase/COGS metrics
            F.coalesce(col("purchase_order_count"), F.lit(0)).alias("purchase_order_count"),
            F.coalesce(col("total_units_purchased"), F.lit(0)).alias("total_units_purchased"),
            F.coalesce(col("total_purchase_cost"), F.lit(0)).alias("total_purchase_cost"),
            F.coalesce(col("avg_purchase_unit_cost"), col("standard_cost")).alias("avg_cogs_per_unit"),
            F.coalesce(col("min_purchase_unit_cost"), col("standard_cost")).alias("min_cogs_per_unit"),
            F.coalesce(col("max_purchase_unit_cost"), col("standard_cost")).alias("max_cogs_per_unit"),
            F.coalesce(col("purchase_cost_std_dev"), F.lit(0)).alias("cogs_std_dev"),
            F.coalesce(col("vendor_count"), F.lit(0)).alias("vendor_count")
        )
        # Calculate profitability metrics
        .withColumn("estimated_total_cogs",
            col("avg_cogs_per_unit") * col("total_units_sold")
        )
        .withColumn("gross_profit",
            col("total_sales_revenue") - col("estimated_total_cogs")
        )
        .withColumn("gross_margin_pct",
            F.when(col("total_sales_revenue") > 0,
                (col("gross_profit") / col("total_sales_revenue") * 100)
            ).otherwise(None)
        )
        .withColumn("gross_margin_per_unit",
            col("avg_selling_price") - col("avg_cogs_per_unit")
        )
        .withColumn("markup_pct",
            F.when(col("avg_cogs_per_unit") > 0,
                ((col("avg_selling_price") - col("avg_cogs_per_unit")) / col("avg_cogs_per_unit") * 100)
            ).otherwise(None)
        )
        .withColumn("revenue_per_customer",
            F.when(col("unique_customers") > 0,
                col("total_sales_revenue") / col("unique_customers")
            ).otherwise(None)
        )
        .withColumn("units_per_order",
            F.when(col("sales_order_count") > 0,
                col("total_units_sold") / col("sales_order_count")
            ).otherwise(None)
        )
        # Profitability classification
        .withColumn("profitability_tier",
            F.when(col("gross_margin_pct").isNull(), "No Data")
            .when(col("gross_margin_pct") >= 50, "High Margin")
            .when(col("gross_margin_pct") >= 30, "Healthy")
            .when(col("gross_margin_pct") >= 15, "Moderate")
            .when(col("gross_margin_pct") >= 0, "Low Margin")
            .otherwise("Unprofitable")
        )
        .withColumn("price_competitiveness",
            F.when(col("avg_selling_price") < col("list_price") * 0.8, "Heavily Discounted")
            .when(col("avg_selling_price") < col("list_price") * 0.95, "Discounted")
            .when(col("avg_selling_price") >= col("list_price"), "Full Price")
            .otherwise("Near List Price")
        )
    )
    
    # Add ranking by profitability within category
    window_category = Window.partitionBy("category_name", "sales_year").orderBy(col("gross_profit").desc())
    
    gold_product_profitability = (gold_product_profitability
        .withColumn("profit_rank_in_category", F.dense_rank().over(window_category))
    )
    
    # Write to Gold
    (gold_product_profitability
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.{GOLD_SCHEMA}.gold_product_profitability")
    )
    
    duration = time.time() - start_time
    row_count = gold_product_profitability.count()
    
    transformation_results.append({
        "table": "gold_product_profitability",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_product_profitability created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({
        "table": "gold_product_profitability",
        "status": "failed",
        "row_count": 0,
        "duration_seconds": round(time.time() - start_time, 2)
    })

In [0]:
# Reorder Point Alerts and Stock Level Monitoring
print("\n" + "=" * 80)
print("⚠️ BUILDING gold_reorder_alerts")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_sales = spark.table(f"workspace.{SILVER_SCHEMA}.fact_sales")
    fact_inventory = spark.table(f"workspace.{SILVER_SCHEMA}.fact_inventory")
    fact_purchases = spark.table(f"workspace.{SILVER_SCHEMA}.fact_purchases")
    dim_product = spark.table(f"workspace.{SILVER_SCHEMA}.dim_product").filter("is_current = true")
    
    # Calculate recent sales demand (last 90 days)
    recent_sales_demand = (fact_sales
        .filter(col("order_date") >= F.date_sub(F.current_date(), 90))
        .groupBy("product_id")
        .agg(
            F.sum("order_quantity").alias("units_sold_90d"),
            F.avg("order_quantity").alias("avg_order_size"),
            F.countDistinct("sales_order_id").alias("order_count_90d"),
            F.max("order_date").alias("last_sale_date")
        )
        .withColumn("daily_demand_rate", col("units_sold_90d") / 90)
    )
    
    # Calculate current stock levels from latest inventory transactions
    current_stock_window = Window.partitionBy("product_id").orderBy(col("transaction_date").desc())
    
    current_stock = (fact_inventory
        .filter(col("is_archived") == False)
        .withColumn("stock_change",
            F.when(col("transaction_type").isin(["W", "P"]), col("quantity"))
            .otherwise(-col("quantity"))
        )
        .withColumn("row_num", F.row_number().over(current_stock_window))
        .filter(col("row_num") == 1)
        .groupBy("product_id")
        .agg(
            F.sum("stock_change").alias("estimated_current_stock"),
            F.max("transaction_date").alias("last_inventory_date")
        )
    )
    
    # Calculate average lead time from recent purchases
    purchase_lead_times = (fact_purchases
        .filter(col("order_date") >= F.date_sub(F.current_date(), 180))
        .withColumn("lead_time_days", F.datediff(col("ship_date"), col("order_date")))
        .groupBy("product_id")
        .agg(
            F.avg("lead_time_days").alias("avg_lead_time_days"),
            F.max("lead_time_days").alias("max_lead_time_days"),
            F.max("order_date").alias("last_purchase_date")
        )
    )
    
    # Combine all metrics and calculate reorder recommendations
    gold_reorder_alerts = (dim_product
        .select(
            "product_id",
            "product_name",
            "product_number",
            "category_name",
            "subcategory_name",
            "standard_cost",
            "list_price"
        )
        .join(recent_sales_demand, "product_id", "left")
        .join(current_stock, "product_id", "left")
        .join(purchase_lead_times, "product_id", "left")
        .withColumn("estimated_current_stock", F.coalesce(col("estimated_current_stock"), F.lit(0)))
        .withColumn("daily_demand_rate", F.coalesce(col("daily_demand_rate"), F.lit(0)))
        .withColumn("avg_lead_time_days", F.coalesce(col("avg_lead_time_days"), F.lit(14)))
        .withColumn("max_lead_time_days", F.coalesce(col("max_lead_time_days"), F.lit(21)))
        
        # Calculate safety stock (assuming 7 days of extra buffer)
        .withColumn("safety_stock",
            col("daily_demand_rate") * 7
        )
        
        # Calculate reorder point (demand during lead time + safety stock)
        .withColumn("reorder_point",
            (col("daily_demand_rate") * col("avg_lead_time_days")) + col("safety_stock")
        )
        
        # Calculate days of stock remaining
        .withColumn("days_of_stock_remaining",
            F.when(col("daily_demand_rate") > 0,
                col("estimated_current_stock") / col("daily_demand_rate")
            ).otherwise(999)
        )
        
        # Calculate suggested order quantity (Economic Order Quantity simplified)
        .withColumn("suggested_order_quantity",
            F.ceil(col("daily_demand_rate") * 30)  # 30 days of demand
        )
        
        # Stock status classification
        .withColumn("stock_status",
            F.when(col("estimated_current_stock") <= 0, "Out of Stock")
            .when(col("estimated_current_stock") < col("reorder_point"), "Below Reorder Point")
            .when(col("days_of_stock_remaining") < 7, "Critical Low")
            .when(col("days_of_stock_remaining") < 14, "Low Stock")
            .when(col("days_of_stock_remaining") < 30, "Adequate")
            .when(col("days_of_stock_remaining") < 60, "Good")
            .otherwise("Overstocked")
        )
        
        # Priority flag for reorder
        .withColumn("reorder_priority",
            F.when(col("stock_status") == "Out of Stock", "Urgent")
            .when(col("stock_status") == "Below Reorder Point", "High")
            .when(col("stock_status") == "Critical Low", "High")
            .when(col("stock_status") == "Low Stock", "Medium")
            .otherwise("Low")
        )
        
        # Calculate estimated stockout date
        .withColumn("estimated_stockout_date",
            F.when(col("daily_demand_rate") > 0,
                F.date_add(F.current_date(), col("days_of_stock_remaining").cast("int"))
            ).otherwise(None)
        )
        
        # Calculate inventory value at cost
        .withColumn("inventory_value_at_cost",
            col("estimated_current_stock") * col("standard_cost")
        )
        
        .select(
            "product_id",
            "product_name",
            "product_number",
            "category_name",
            "subcategory_name",
            "estimated_current_stock",
            "units_sold_90d",
            "daily_demand_rate",
            "avg_lead_time_days",
            "max_lead_time_days",
            "safety_stock",
            "reorder_point",
            "days_of_stock_remaining",
            "suggested_order_quantity",
            "stock_status",
            "reorder_priority",
            "estimated_stockout_date",
            "last_sale_date",
            "last_inventory_date",
            "last_purchase_date",
            "standard_cost",
            "list_price",
            "inventory_value_at_cost",
            F.current_timestamp().alias("analysis_timestamp")
        )
    )
    
    # Write to Gold
    (gold_reorder_alerts
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.{GOLD_SCHEMA}.gold_reorder_alerts")
    )
    
    duration = time.time() - start_time
    row_count = gold_reorder_alerts.count()
    
    transformation_results.append({
        "table": "gold_reorder_alerts",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_reorder_alerts created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({
        "table": "gold_reorder_alerts",
        "status": "failed",
        "row_count": 0,
        "duration_seconds": round(time.time() - start_time, 2)
    })

In [0]:
# Summary
print("\n" + "=" * 80)
print("📊 INVENTORY & SUPPLY CHAIN ANALYTICS SUMMARY")
print("=" * 80)

success_count = sum(1 for r in transformation_results if r["status"] == "success")
failed_count = sum(1 for r in transformation_results if r["status"] == "failed")
total_rows = sum(r["row_count"] for r in transformation_results)

print(f"\nGold Tables Created: {len(transformation_results)}")
print(f"  ✅ Success: {success_count}")
print(f"  ❌ Failed: {failed_count}")
print(f"Total Rows Created: {total_rows:,}")

summary_df = spark.createDataFrame([Row(**r) for r in transformation_results])
print("\nDetailed Results:")
display(summary_df.orderBy("status", "table"))

print("\n" + "=" * 80)
if failed_count == 0:
    print("✅ GOLD INVENTORY & SUPPLY CHAIN ANALYTICS COMPLETED SUCCESSFULLY")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit("SUCCESS")
else:
    print("⚠️ GOLD INVENTORY ANALYTICS COMPLETED WITH ERRORS")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit(f"PARTIAL_SUCCESS: {failed_count} tables failed")